In [1]:
# =============================================================================
# GUS03: Optimizing geoTERYT Database
# =============================================================================
# 
# ...
#
# =============================================================================

# STEP 1: Imports and Path Setup
import os
import sys
from pathlib import Path
import importlib
import gc
import random

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# Find the repository root
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists():
            return p
    return start

repo_root = find_repo_root()
tools_path = repo_root / 'Code' / 'tools'
if str(tools_path) not in sys.path:
    sys.path.insert(0, str(tools_path))

import geoTERYT_db as gtdb
importlib.reload(gtdb)

# Paths
data_root = repo_root.parent.parent / 'Data'
geo_root = data_root / 'Geospatial'
gus_root = data_root / 'GUS'
json_root = gus_root / 'data' / 'extracted'

print(f"Repository root: {repo_root}")
print(f"GUS root: {gus_root}")
print(f"JSON root: {json_root}")

Repository root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper
GUS root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/GUS
JSON root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/GUS/data/extracted


In [2]:
# =============================================================================
# STEP 2: Load Database
# =============================================================================
complete_db_path = geo_root / 'geoteryt_complete_final.pkl'
db = gtdb.load_complete_database(complete_db_path)
db.print_summary()
print(f"\nData summary: {db.get_data_summary()}")

gc.collect()

Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl...
  Database version: 4.2
  ✓ Restored old voivodships: 49 rows
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4612 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3661
  ✓ Records with old_woj: 4104
  ✓ Records with data: 4584
  ✓ Records with cross tables: 4584
  ✓ Records with population data: 4582
  ✓ Records with pop_class: 3411
GeoTERYT Database Summary (v3.0)
Total records:           4,612
Year range:              1999 - 2024
------------------------------------------------------------
Administrative levels:
  Voivodeships (2):      16
  Powiats (5):           382
  Gminas (6):            4162
------------------------------------------------------------
Change tracking:
  Records with cha

0

In [3]:
# =============================================================================
# STEP 3: Run performance optimization methods and save optimized database
# =============================================================================
import sys
import time

# --- Before: measure geometry statistics ---
n_gdfs = len(db._geometries)
total_geom_slots = sum(len(gdf) for gdf in db._geometries.values())
n_non_null_geoms = sum(
    gdf.geometry.notna().sum() for gdf in db._geometries.values()
)
n_records_with_geom = sum(1 for r in db._records.values() if r.has_geometry)

print("=" * 60)
print("BEFORE OPTIMIZATION")
print("=" * 60)
print(f"  Geometry GDFs:             {n_gdfs} years")
print(f"  Total GDF geometry slots:  {total_geom_slots:,}")
print(f"  Non-null GDF geometries:   {n_non_null_geoms:,}")
print(f"  Records with geometry:     {n_records_with_geom:,}")

# Count unique geometry objects (by Python id) across GDFs + records
geom_ids_before = set()
for gdf in db._geometries.values():
    for geom in gdf.geometry:
        if geom is not None:
            geom_ids_before.add(id(geom))
for r in db._records.values():
    if r.geometry is not None:
        geom_ids_before.add(id(r.geometry))
print(f"  Unique Python geometry objects: {len(geom_ids_before):,}")
print()

# --- Run reorganize_geometries ---
print("Running db.reorganize_geometries()...")
print("-" * 60)
stats = db.reorganize_geometries(tolerance=0.01, verbose=True)
print("-" * 60)
print()

# --- After: measure geometry statistics ---
n_non_null_after = sum(
    gdf.geometry.notna().sum() for gdf in db._geometries.values()
)
geom_ids_after = set()
for gdf in db._geometries.values():
    for geom in gdf.geometry:
        if geom is not None:
            geom_ids_after.add(id(geom))
for r in db._records.values():
    if r.geometry is not None:
        geom_ids_after.add(id(r.geometry))

print("=" * 60)
print("AFTER OPTIMIZATION")
print("=" * 60)
print(f"  Non-null GDF geometries:   {n_non_null_after:,}  (was {n_non_null_geoms:,})")
print(f"  Geometry store size:       {len(db._geometry_store):,} unique geometries")
print(f"  Unique Python geom objects: {len(geom_ids_after):,}  (was {len(geom_ids_before):,})")
print(f"  Records with geometry:     {sum(1 for r in db._records.values() if r.has_geometry):,}")
reduction_pct = (1 - len(geom_ids_after) / max(len(geom_ids_before), 1)) * 100
print(f"  Object count reduction:    {reduction_pct:.1f}%")
print()

# Force garbage collection to free old geometry objects
del geom_ids_before, geom_ids_after
gc.collect()

# --- Verify: spot-check that geometries still resolve correctly ---
print("Verification: spot-checking geometry access...")
sample_records = [r for r in db._records.values() if r.has_geometry]
random.seed(42)
check_sample = random.sample(sample_records, min(50, len(sample_records)))
all_ok = True
for r in check_sample:
    g = db.get_geometry(r.teryt_id)
    if g is None or g.is_empty:
        print(f"  WARNING: {r.teryt_id} geometry is None/empty!")
        all_ok = False
if all_ok:
    print(f"  ✓ All {len(check_sample)} spot-checked records have valid geometries")

# Also verify _get_row_geometry resolves across GDFs
sample_years = list(db._geometries.keys())[:3]
for yr in sample_years:
    gdf = db._geometries[yr]
    null_rows = gdf[gdf.geometry.isna() & gdf['_geom_hash'].notna()]
    if len(null_rows) > 0:
        test_row = null_rows.iloc[0]
        resolved = db._get_row_geometry(test_row)
        if resolved is not None:
            print(f"  ✓ Year {yr}: GDF reference resolution works ({len(null_rows)} referenced rows)")
        else:
            print(f"  WARNING: Year {yr}: reference resolution failed!")
            all_ok = False
print()

# --- Save optimized database ---
optimized_db_path = geo_root / 'geoteryt_O.pkl'
print(f"Saving optimized database to {optimized_db_path}...")
t0 = time.time()
db.save_complete(optimized_db_path, verbose=True)
save_time = time.time() - t0
print(f"  Save time: {save_time:.1f}s")
print()

# --- Reload and verify ---
print("Reloading optimized database to verify...")
t0 = time.time()
db2 = gtdb.load_complete_database(optimized_db_path, verbose=True)
load_time = time.time() - t0
print(f"  Load time: {load_time:.1f}s")
print()

# Quick integrity checks
n_rec_orig = len(db._records)
n_rec_loaded = len(db2._records)
n_geom_orig = sum(1 for r in db._records.values() if r.has_geometry)
n_geom_loaded = sum(1 for r in db2._records.values() if r.has_geometry)
n_data_orig = sum(1 for r in db._records.values() if r.has_data)
n_data_loaded = sum(1 for r in db2._records.values() if r.has_data)

print("=" * 60)
print("INTEGRITY CHECK")
print("=" * 60)
print(f"  Records:      {n_rec_loaded:,}  (original: {n_rec_orig:,})  {'✓' if n_rec_loaded == n_rec_orig else '✗ MISMATCH!'}")
print(f"  With geometry: {n_geom_loaded:,}  (original: {n_geom_orig:,})  {'✓' if n_geom_loaded == n_geom_orig else '✗ MISMATCH!'}")
print(f"  With data:     {n_data_loaded:,}  (original: {n_data_orig:,})  {'✓' if n_data_loaded == n_data_orig else '✗ MISMATCH!'}")
print(f"  Geom store:    {len(db2._geometry_store):,}  {'✓' if len(db2._geometry_store) == len(db._geometry_store) else '✗ MISMATCH!'}")
print(f"  Reorganized:   {db2._geometries_reorganized}  {'✓' if db2._geometries_reorganized else '✗ NOT SET!'}")

# Spot-check data values
random.seed(123)
sample_data_records = [r for r in db._records.values() if r.has_data]
data_check = random.sample(sample_data_records, min(20, len(sample_data_records)))
data_ok = True
for r_orig in data_check:
    r_loaded = db2._records[r_orig.teryt_id]
    for key, ds_orig in r_orig.data.items():
        ds_loaded = r_loaded.data.get(key)
        if ds_loaded is None:
            print(f"  WARNING: Missing data series {key} for {r_orig.teryt_id}")
            data_ok = False
            continue
        if not ds_orig.values.equals(ds_loaded.values):
            # Check with NaN-aware comparison
            orig_vals = ds_orig.values.dropna()
            loaded_vals = ds_loaded.values.dropna()
            if not orig_vals.equals(loaded_vals):
                print(f"  WARNING: Data mismatch for {r_orig.teryt_id} {key}")
                data_ok = False
if data_ok:
    print(f"  ✓ Data integrity verified for {len(data_check)} records")

# Clean up reload
del db2
gc.collect()
print("\n✓ Optimization complete!")

BEFORE OPTIMIZATION
  Geometry GDFs:             15 years
  Total GDF geometry slots:  37,175
  Non-null GDF geometries:   37,175
  Records with geometry:     3,661
  Unique Python geometry objects: 40,836

Running db.reorganize_geometries()...
------------------------------------------------------------
Reorganizing geometries across 15 years (37,175 total geometry slots)...
  Pass 1 (WKB hash): 15,594 unique, 21,581 duplicates identified
  Pass 2 (spatial intersection, tol=0.01m): checking 15,594 canonical geometries...
    Checked 326,007 candidate pairs, found 2307 additional topological matches
  Geometry store: 13,287 unique geometries
  GDF cleanup: kept 13,287 canonical, cleared 23,888 duplicates
  Record geometries: 3,612/3,661 now share canonical objects
  ✓ Reorganization complete in 7.8s
    Unique geometries in store: 13,287
    GDF slots freed: 23,888 / 37,175
------------------------------------------------------------

AFTER OPTIMIZATION
  Non-null GDF geometries:   13,

In [4]:
# =============================================================================
# STEP 2: Load Database
# =============================================================================
complete_db_path = geo_root / 'geoteryt_O.pkl'
db = gtdb.load_complete_database(complete_db_path)
db.print_summary()
print(f"\nData summary: {db.get_data_summary()}")

gc.collect()

Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_O.pkl...
  Database version: 4.3
  ✓ Restored old voivodships: 49 rows
  ✓ Restored geometry store: 13,287 unique geometries
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4612 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3661
  ✓ Records with old_woj: 4104
  ✓ Records with data: 4584
  ✓ Records with cross tables: 4584
  ✓ Records with population data: 4582
  ✓ Records with pop_class: 3411
GeoTERYT Database Summary (v3.0)
Total records:           4,612
Year range:              1999 - 2024
------------------------------------------------------------
Administrative levels:
  Voivodeships (2):      16
  Powiats (5):           382
  Gminas (6):            4162
-------------------------------------------------------

0

In [5]:
db.to_dataframe()

,teryt_id,name,name_dod,level,kind,woj,pow,gmi,rodz,years_valid,...,code_by_year,has_data,n_data_series,data_subjects,has_cross_tables,n_cross_tables,cross_table_subjects,has_pop,pop_years,has_pop_class
0,0200000,DOLNOŚLĄSKIE,województwo,2,NaN,02,00,00,0,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",...,"{1999: '0200000', 2000: '0200000', 2001: '0200...",True,993,"[P4315, P4253, P2403, M_pop__age_educ, M_pop__...",True,25,"[P2137, P2884, P2885, P2883, P2887, P2114, P24...",True,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",False
1,0201000,bolesławiecki,powiat,5,NaN,02,01,00,0,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",...,"{1999: '0201000', 2000: '0201000', 2001: '0201...",True,1000,"[P4315, P4253, P2403, M_pop__age_educ, M_educ_...",True,25,"[P2137, P2884, P2885, P2883, P2887, P2114, P24...",True,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",False
2,0201011,Bolesławiec,gmina miejska,6,urban,02,01,01,1,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",...,"{1999: '0201011', 2000: '0201011', 2001: '0201...",True,546,"[P4315, P4253, M_educ_sex_2000, M_hh_size_2000...",True,24,"[P2137, P2884, P2885, P2883, P2887, P2114, P24...",True,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",True
3,0201022,Bolesławiec,gmina wiejska,6,rural,02,01,02,2,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",...,"{1999: '0201022', 2000: '0201022', 2001: '0201...",True,546,"[P4315, P4253, M_educ_sex_2000, M_hh_size_2000...",True,24,"[P2137, P2884, P2885, P2883, P2887, P2114, P24...",True,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",True
4,0201032,Gromadka,gmina wiejska,6,rural,02,01,03,2,"[1999, 2000, 2001, 2002, 2003, 2004, 2005, 200...",...,"{1999: '0201032', 2000: '0201032', 2001: '0201...",True,546,"[P4315, P4253, M_educ_sex_2000, M_hh_size_2000...",True,24,"[P2137, P2884, P2885, P2883, P2887, P2114, P24...",True,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4607,5300000,Zielonogórskie,None,2,0,53,00,00,0,[],...,{},True,107,"[M_age_sex, M_age_1990, H_age_sex]",True,3,"[M_age_sex, M_age_1990, H_age_sex]",True,[1988],False
4608,5200000,Łomżyńskie,None,2,0,52,00,00,0,[],...,{},True,107,"[M_age_sex, M_age_1990, H_age_sex]",True,3,"[M_age_sex, M_age_1990, H_age_sex]",True,[1988],False
4609,5100000,Łódzkie,None,2,0,51,00,00,0,[],...,{},True,107,"[M_age_sex, M_age_1990, H_age_sex]",True,3,"[M_age_sex, M_age_1990, H_age_sex]",True,[1988],False
4610,1300000,Warszawski stołeczny,None,2,0,13,00,00,0,[],...,{},True,20,"[P2350, M_educ_2000, M_pop__educ, P4092]",True,4,"[P2350, P4092, M_pop__educ, M_educ_2000]",False,[],False
